# 🎁 Disponibilizando um Minicurso de SQL

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-sql%20%7C%20referência-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Iniciante-green)
![Pré-req](https://img.shields.io/badge/pré--req-nenhum-purple)
![Biblioteca](https://img.shields.io/badge/requer-sqlite3%20(nativo)-orange)

> Até aqui, todo SQL usado foi só o suficiente pra fazer o `pyodbc`/`mysql-connector` funcionar — sem entrar a fundo na linguagem em si. Este notebook não é uma aula nova: é um **cheat sheet de referência rápida** dos comandos SQL mais usados, pra consultar sempre que precisar, rodando ao vivo com o `sqlite3` (que já vem com o Python, sem precisar de nenhum servidor).

## 📋 Conteúdo

1. [Consulta (SELECT)](#-1-consulta-select)
2. [Filtros e Ordenação](#-2-filtros-e-ordenação)
3. [Agregações (GROUP BY)](#-3-agregações-group-by)
4. [Junções (JOIN)](#-4-junções-join)
5. [Escrita (INSERT, UPDATE, DELETE)](#-5-escrita-insert-update-delete)


## 📖 1. Consulta (SELECT)

| Comando 🔑 | Papel 🔓 |
|---|---|
| `SELECT * FROM tabela` | traz todas as colunas |
| `SELECT coluna1, coluna2 FROM tabela` | traz só as colunas escolhidas |
| `SELECT DISTINCT coluna FROM tabela` | traz valores sem repetição |
| `SELECT TOP 5 * FROM tabela` (SQL Server) / `... LIMIT 5` (MySQL, SQLite) | limita a quantidade de linhas |

A base de exemplo abaixo — uma tabela `Vendas` bem pequena, criada com `sqlite3` — é usada em todas as próximas seções.

In [1]:
import sqlite3
import pandas as pd
from cores import *

conexao = sqlite3.connect(":memory:")
cursor = conexao.cursor()

cursor.execute("""
CREATE TABLE Vendas (
    Id INTEGER PRIMARY KEY,
    Produto TEXT,
    Categoria TEXT,
    Quantidade INTEGER,
    ValorUnitario REAL,
    Vendedor TEXT
)
""")
cursor.executemany(
    "INSERT INTO Vendas (Produto, Categoria, Quantidade, ValorUnitario, Vendedor) VALUES (?, ?, ?, ?, ?)",
    [
        ("Notebook", "Eletrônicos", 2, 4200.0, "Ana"),
        ("Mouse", "Eletrônicos", 5, 89.9, "Bruno"),
        ("Cadeira", "Móveis", 1, 899.0, "Ana"),
        ("Mesa", "Móveis", 1, 650.0, "Carla"),
        ("Monitor", "Eletrônicos", 3, 1100.0, "Bruno"),
    ]
)
conexao.commit()

pd.read_sql("SELECT * FROM Vendas", conexao)


,Id,Produto,Categoria,Quantidade,ValorUnitario,Vendedor
0,1,Notebook,Eletrônicos,2,4200.0,Ana
1,2,Mouse,Eletrônicos,5,89.9,Bruno
2,3,Cadeira,Móveis,1,899.0,Ana
3,4,Mesa,Móveis,1,650.0,Carla
4,5,Monitor,Eletrônicos,3,1100.0,Bruno


## 🔍 2. Filtros e Ordenação

| Comando 🔑 | Papel 🔓 |
|---|---|
| `WHERE coluna = valor` | filtra linhas |
| `WHERE coluna > valor` | comparação numérica |
| `WHERE coluna LIKE '%termo%'` | busca por texto parcial |
| `WHERE coluna IN (a, b, c)` | filtra por uma lista de valores |
| `ORDER BY coluna DESC` | ordena (do maior pro menor) |

In [2]:
pd.read_sql(
    "SELECT Produto, ValorUnitario FROM Vendas WHERE Categoria = 'Eletrônicos' ORDER BY ValorUnitario DESC",
    conexao
)


,Produto,ValorUnitario
0,Notebook,4200.0
1,Monitor,1100.0
2,Mouse,89.9


## 📊 3. Agregações (GROUP BY)

| Função 🔑 | Papel 🔓 |
|---|---|
| `SUM(coluna)` | soma |
| `AVG(coluna)` | média |
| `COUNT(*)` | quantidade de linhas |
| `MAX(coluna)` / `MIN(coluna)` | maior / menor valor |
| `GROUP BY coluna` | agrupa antes de agregar |
| `HAVING` | filtra depois do `GROUP BY` (o `WHERE` não pode filtrar agregação) |

In [3]:
pd.read_sql(
    """SELECT Categoria, SUM(Quantidade * ValorUnitario) AS Faturamento
       FROM Vendas
       GROUP BY Categoria
       ORDER BY Faturamento DESC""",
    conexao
)


,Categoria,Faturamento
0,Eletrônicos,12149.5
1,Móveis,1549.0


## 🔗 4. Junções (JOIN)

Quando o dado está espalhado em mais de uma tabela (ex.: `Vendas` e `Vendedores`), o `JOIN` cruza as duas usando uma coluna em comum.

| Tipo 🔑 | Traz 🔓 |
|---|---|
| `INNER JOIN` | só as linhas que têm correspondência nas duas tabelas |
| `LEFT JOIN` | todas as linhas da tabela da esquerda, mesmo sem correspondência |

In [4]:
cursor.execute("CREATE TABLE Vendedores (Nome TEXT, Regiao TEXT)")
cursor.executemany(
    "INSERT INTO Vendedores VALUES (?, ?)",
    [("Ana", "Nordeste"), ("Bruno", "Sudeste"), ("Carla", "Sul")]
)
conexao.commit()

pd.read_sql(
    """SELECT V.Produto, V.Vendedor, Vd.Regiao
       FROM Vendas V
       INNER JOIN Vendedores Vd ON V.Vendedor = Vd.Nome""",
    conexao
)


,Produto,Vendedor,Regiao
0,Notebook,Ana,Nordeste
1,Mouse,Bruno,Sudeste
2,Cadeira,Ana,Nordeste
3,Mesa,Carla,Sul
4,Monitor,Bruno,Sudeste


## ✍️ 5. Escrita (INSERT, UPDATE, DELETE)

| Comando 🔑 | Sintaxe 🔓 |
|---|---|
| Inserir | `INSERT INTO tabela (col1, col2) VALUES (v1, v2)` |
| Atualizar | `UPDATE tabela SET col1 = novo_valor WHERE condição` |
| Apagar | `DELETE FROM tabela WHERE condição` |
| Criar tabela | `CREATE TABLE tabela (col1 TIPO, col2 TIPO, ...)` |
| Apagar tabela | `DROP TABLE tabela` |

> 🎁 **Bônus:** a Hashtag disponibiliza um minicurso completo de SQL puro (sem Python), cobrindo esses comandos com muito mais profundidade — consulte a área de bônus da plataforma do curso.

In [5]:
conexao.close()
print(f"{VerdeClaro}Cheat sheet concluído — este notebook não precisa ser mantido aberto.{Reset}")


Cheat sheet concluído — este notebook não precisa ser mantido aberto.


Com o vocabulário básico de SQL revisado, o último notebook do módulo muda de abordagem: em vez de escrever SQL na mão, o **SQLAlchemy** cria o banco inteiro (tabelas incluídas) só com classes Python.

> ▶️ Próximo notebook: **SQLAlchemy - Como criar Banco de Dados usando apenas Python**.